Week04_Sat

Link Github: https://github.com/BM910/AI-ARIN330585

Ma trận 2 chiều "room" biểu diễn sàn nhà:
    0 - Không có bụi, có thể đi qua được.
    1 - Có bụi, có thể đi qua được.
    2 - Vật cản, không đi qua được.
    3 - Máy hút bụi.

In [2]:
from collections import deque
import heapq
import tkinter as tk
import random

current_room = None

class Node:
    def __init__(self, state, parent=None, action=None, depth=0, g_value=0, use_heuristic=False):
        self.state = tuple(tuple(row) for row in state) if isinstance(state, list) else state
        self.parent = parent
        self.action = action
        self.depth = depth

        self.g_value = g_value
        self.h_value = self.heu_dirty_value() if use_heuristic else 0
        self.f_value = self.g_value + self.h_value

    def __str__(self):
        state = ""
        for row in self.state:
            state += str(row) + '\n'
        move_and_cost = f"Moved {self.action}"
        return state + move_and_cost

    def heu_dirty_value(self):
        total_dirty_value = 0
        dirty_tile_value = 1
        for row in self.state:
            total_dirty_value += dirty_tile_value * (row.count(1))
        return total_dirty_value

    def is_clean(self):
        for row in self.state:
            if 1 in row:
                return False
        return True

    def find_roomba(self):
        for i, row in enumerate(self.state):
            if 3 in row:
                return (i, row.index(3))
        return None

    def get_moves(self):
        x, y = self.find_roomba()
        full_moves = [(-1, 0, "UP"), (1, 0, "DOWN"), (0, -1, "LEFT"), (0, 1, "RIGHT")]
        # prioritized_moves = [] # prioritize
        valid_moves = []

        for dx, dy, action in full_moves:
            if self.check_valid_move(x, y, dx, dy):
                nx, ny = x + dx, y + dy
                if self.state[nx][ny] == 1:
                    action += f" and cleaned [{nx}][{ny}]"
                    # prioritized_moves.append((dx, dy, action)) # prioritize
                valid_moves.append((dx, dy, action))
        
        # random.shuffle(valid_moves) # randomize
        # valid_moves = prioritized_moves + valid_moves # prioritize
        return valid_moves
                    
    def check_valid_move(self, x, y, dx, dy):
        nx, ny = x + dx, y + dy
        if 0 <= nx < len(self.state) and 0 <= ny < len(self.state[0]) and self.state[nx][ny] != 2:
            return True
        return False
    
    def generate_new_state(self, dx, dy):
        x, y = self.find_roomba()
        nx, ny = x + dx, y + dy
        new_state = [list(row) for row in self.state]
        new_state[x][y] = 0
        new_state[nx][ny] = 3
        return tuple(tuple(row) for row in new_state)
    
    def get_parent_list(self):
        path = [self]
        while self.parent:
            path.append(self.parent)
            self = self.parent
        return path[::-1]

def bfs1(initial_state):
    if not initial_state:
        return
    node = Node(initial_state)

    frontier = deque()
    reached = set()
    frontier.append(node)
    frontier_set = set()
    frontier_set.add(initial_state)

    while frontier:
        node = frontier.popleft()
        frontier_set.discard(node.state)
        reached.add(node.state)

        if node.is_clean():
            update_log("BFS_1")
            update_log(f"reached states: {len(reached)}") # stat
            update_log("-"*20)
            return node
        
        moves = node.get_moves()
        
        for dx, dy, action in moves:
            child_node = Node(node.generate_new_state(dx, dy), node, action, node.depth+1)

            if child_node.state not in frontier_set and child_node.state not in reached:
                frontier.append(child_node)
                frontier_set.add(child_node.state)

    return None

def bfs2(initial_state):
    if not initial_state:
        return
    node = Node(initial_state)

    frontier = deque()
    reached = set()
    frontier.append(node)
    frontier_set = set()
    frontier_set.add(initial_state)

    while frontier:
        node = frontier.popleft()
        frontier_set.discard(node.state)
        reached.add(node.state)
        
        moves = node.get_moves()
        
        for dx, dy, action in moves:
            child_node = Node(node.generate_new_state(dx, dy), node, action, node.depth+1)
            if child_node.is_clean():
                update_log("BFS_2")
                update_log(f"reached states: {len(reached)}") # stat
                update_log("-"*20)
                return child_node

            if child_node.state not in frontier_set and child_node.state not in reached:
                frontier.append(child_node)
                frontier_set.add(child_node.state)

    return None

def dfs1(initial_state):
    if not initial_state:
        return
    node = Node(initial_state)

    frontier = []
    reached = set()
    frontier.append(node)
    frontier_set = set()
    frontier_set.add(initial_state)

    while frontier:
        node = frontier.pop()
        frontier_set.discard(node.state)
        reached.add(node.state)

        if node.is_clean():
            update_log("DFS_1")
            update_log(f"reached states: {len(reached)}") # stat
            update_log("-"*20)
            return node
        
        moves = node.get_moves()
        
        for dx, dy, action in moves:
            child_node = Node(node.generate_new_state(dx, dy), node, action, node.depth+1)

            if child_node.state not in frontier_set and child_node.state not in reached:
                frontier.append(child_node)
                frontier_set.add(child_node.state)

    return None

def dfs2(initial_state):
    if not initial_state:
        return
    node = Node(initial_state)

    frontier = []
    reached = set()
    frontier.append(node)
    frontier_set = set()
    frontier_set.add(initial_state)

    while frontier:
        node = frontier.pop()
        frontier_set.discard(node.state)
        reached.add(node.state)
        
        moves = node.get_moves()
        
        for dx, dy, action in moves:
            child_node = Node(node.generate_new_state(dx, dy), node, action, node.depth+1)
            if child_node.is_clean():
                update_log("DFS_2")
                update_log(f"reached states: {len(reached)}") # stat
                update_log("-"*20)
                return child_node

            if child_node.state not in frontier_set and child_node.state not in reached:
                frontier.append(child_node)
                frontier_set.add(child_node.state)

    return None

def ids(initial_state, max_depth=200):
    for depth in range(max_depth):
        result = dls(initial_state, depth)
        if result != "cutoff":
            return result
    return None

def dls(initial_state, limit):
    frontier = []
    node = Node(initial_state, depth=0)
    frontier.append(node)
    result = None

    while frontier:
        node = frontier.pop()

        if node.is_clean():
            update_log("IDS")
            update_log("-"*20)
            return node
        
        if node.depth >= limit:
            result = "cutoff"
            continue

        is_cycle = False
        parent_node = node.parent
        while parent_node:
            if parent_node.state == node.state:
                is_cycle = True
                break
            parent_node = parent_node.parent

        if not is_cycle:
            moves = node.get_moves()
            for dx, dy, action in moves:
                child_node = Node(node.generate_new_state(dx, dy), node, action, node.depth + 1)
                frontier.append(child_node)

    return result

def ucs(initial_state):
    if not initial_state:
        return
    node = Node(initial_state)

    counter = 0
    frontier = [(node.g_value, counter, node)]
    reached = {str(node.state) : 0}

    while frontier:
        current_g_value, _, node = heapq.heappop(frontier)
        if node.is_clean():
            update_log("UCS")
            update_log(f"reached states: {len(reached)}")
            update_log("-"*20)
            return node

        if current_g_value > reached[str(node.state)]:
            continue

        moves = node.get_moves()
        for dx, dy, action in moves:
            child_node = Node(node.generate_new_state(dx, dy), node, action, depth=node.depth+1, g_value=node.g_value+1)
            
            x, y = child_node.find_roomba()
            if node.state[x][y] == 1:
                child_node.g_value -= 1

            child_str = str(child_node.state)
            child_cost = child_node.g_value
            if child_str not in reached or child_cost < reached[child_str]:
                reached[child_str] = child_cost
                counter += 1
                heapq.heappush(frontier, (child_cost, counter, child_node))

    return None

def greedy(initial_state):
    if not initial_state:
        return
    node = Node(initial_state, use_heuristic=True)
    
    counter = 0
    frontier = [(node.h_value, counter, node)]
    reached = set()

    while frontier:
        _, _, node = heapq.heappop(frontier)

        if node.is_clean():
            update_log("GREEDY")
            update_log(f"reached states: {len(reached)}")
            update_log("-"*20)
            return node
        reached.add(node.state)

        moves = node.get_moves()
        for dx, dy, action in moves:
            child_node = Node(node.generate_new_state(dx, dy), node, action, depth=node.depth+1, use_heuristic=True)
            if child_node.state not in reached:
                counter += 1
                heapq.heappush(frontier, (child_node.h_value, counter, child_node))

    return None

def a_star(initial_state):
    if not initial_state:
        return
    node = Node(initial_state, g_value=0, use_heuristic=True)
    
    counter = 0
    frontier = [(node.f_value, counter, node)]
    reached = {str(node.state) : node.g_value}

    while frontier:
        _, _, node = heapq.heappop(frontier)

        if node.is_clean():
            update_log("A_STAR")
            update_log(f"reached states: {len(reached)}")
            update_log("-"*20)
            return node
        
        if node.g_value > reached[str(node.state)]:
            continue

        moves = node.get_moves()
        for dx, dy, action in moves:
            child_node = Node(node.generate_new_state(dx, dy), node, action, depth=node.depth+1 ,g_value=node.g_value+1, use_heuristic=True)
            child_node_str = str(child_node.state)
            if child_node_str not in reached or child_node_str in reached and child_node.g_value < reached[child_node_str]:
                    reached[child_node_str] = child_node.g_value
                    counter += 1
                    heapq.heappush(frontier, (child_node.f_value, counter, child_node))

    return None



def randomize_map():
    global current_room

    rows = random.randint(3, 8)
    cols = rows

    choices = [0, 1, 2]
    weights = [0.60, 0.20, 0.20]

    current_room = []
    for r in range(rows):
        row_tiles = random.choices(choices, weights=weights, k=cols)
        current_room.append(row_tiles)
        
    roomba_placed = False
    while not roomba_placed:
        random_row = random.randint(0, rows - 1)
        random_col = random.randint(0, cols - 1)
        
        if current_room[random_row][random_col] in (0, 1):
            current_room[random_row][random_col] = 3
            roomba_placed = True

    current_room = tuple(tuple(row) for row in current_room)
    draw_matrix_on_map(current_room)

def draw_matrix_on_map(matrix):
    # Clear anything previously drawn on the map canvas
    map_canvas.delete("all")
    
    rows = len(matrix)
    cols = len(matrix[0])
    
    # Dynamically scale cell size to perfectly fit your 400x400 map area
    cell_width = 500 // cols
    cell_height = 500 // rows
    
    # Map out the color scheme per rule
    colors = {
        0: "lightgray",  # Clean, passable floor
        1: "#7A592F",    # Dirty floor
        2: "#1B1A1A",    # Obstacle
        3: "green"     # Roomba
    }
    
    for r in range(rows):
        for c in range(cols):
            val = matrix[r][c]
            color = colors.get(val, "white")
            
            # Calculate pixel bounds for each grid coordinate
            x1 = c * cell_width
            y1 = r * cell_height
            x2 = x1 + cell_width
            y2 = y1 + cell_height
            
            # Draw the square onto the canvas
            map_canvas.create_rectangle(x1, y1, x2, y2, fill=color, outline="black", width=2)
            
            # Optional visual aid: Draw a distinct circle if it's the Roomba
            if val == 3:
                map_canvas.create_oval(x1+5, y1+5, x2-5, y2-5, fill="gray", outline="white", width=2)

def run_and_animate_algo(algo_func):
    global current_room
    if current_room is None:
        update_log("Please generate a map first using Randomize!")
        return

    final_node = algo_func(current_room)
    
    if final_node is None:
        update_log("No solution found to clean the room!")
        return
        
    # path_timeline: [(state_1, action_1), (state_2, action_2), ...]
    parent_list = final_node.get_parent_list()
    path_timeline = []
    for node in parent_list:
        path_timeline.append((node.state, str(node.action)))
    
    animate_steps(path_timeline, 0)

def animate_steps(path_timeline, step_index):
    if step_index >= len(path_timeline):
        update_log("Animation complete! Room is clean.\n")
        return

    room_state, action_text = path_timeline[step_index]
    
    draw_matrix_on_map(room_state)
    
    update_log("Roomba moved " + action_text)
    update_log("-"*20)

    window.after(150, lambda: animate_steps(path_timeline, step_index + 1))

def log_path_timeline(path_timeline):
    moves = ""
    for state, step in path_timeline:
        step = step.split(maxsplit=1)[0]
        moves += step + " -> "
    moves += "DONE\n"
    update_log(f"Path: {moves}")

def update_log(message="", clear=False):
    if clear:
        log_text.delete("1.0", tk.END)
    if message != "":
        log_text.insert(tk.END, message + "\n")
    
    log_text.see(tk.END)

# --- WINDOW SETUP ---
window = tk.Tk()
window.title("Layout Fix")

window.columnconfigure(0, weight=15)
window.columnconfigure(1, weight=40)
window.columnconfigure(2, weight=25)
window.rowconfigure(0, weight=1)
window.rowconfigure(1, weight=0)

# Sidebar Frame
frm_algo_options = tk.Frame(bg="gray", width=150, height=500, relief=tk.SUNKEN, border=5)
frm_algo_options.grid(column=0, row=0, sticky="nsew")
frm_algo_options.grid_propagate(False)
frm_algo_options.columnconfigure(0, weight=1)

# Map Frame (Replacing generic frame with a Canvas widget for pixel manipulation)
map_canvas = tk.Canvas(master=window, bg="gray", width=500, height=500, relief=tk.SUNKEN, border=5)
map_canvas.grid(column=1, row=0, sticky="nsew")
map_canvas.grid_propagate(False)

# Log Frame
frm_state_log = tk.Frame(bg="gray", width=150, height=500, relief=tk.SUNKEN, border=5)
frm_state_log.grid(column=2, row=0, sticky="nsew")
frm_state_log.grid_propagate(False)

# Actions Frame
frm_actions = tk.Frame(bg="darkgray", width=800, height=50, relief=tk.SUNKEN, border=5)
frm_actions.grid(column=0, columnspan=3, row=1, sticky="nsew")
frm_actions.grid_propagate(False)


# --- BUTTONS ---
btn_font = ("Helvetica", 12, "bold")

btn_bfs1 = tk.Button(frm_algo_options, text="BFS_1", bg="white", font=btn_font, command=lambda: run_and_animate_algo(bfs1))
btn_bfs1.grid(column=0, row=0, sticky="ew", padx=10, pady=5)

btn_bfs2 = tk.Button(frm_algo_options, text="BFS_2", bg="white", font=btn_font, command=lambda: run_and_animate_algo(bfs2))
btn_bfs2.grid(column=0, row=1, sticky="ew", padx=10, pady=5)

btn_dfs1 = tk.Button(frm_algo_options, text="DFS_1", bg="white", font=btn_font, command=lambda: run_and_animate_algo(dfs1))
btn_dfs1.grid(column=0, row=2, sticky="ew", padx=10, pady=5)

btn_dfs2 = tk.Button(frm_algo_options, text="DFS_2", bg="white", font=btn_font, command=lambda: run_and_animate_algo(dfs2))
btn_dfs2.grid(column=0, row=3, sticky="ew", padx=10, pady=5)

btn_ids = tk.Button(frm_algo_options, text="IDS", bg="white", font=btn_font, command=lambda: run_and_animate_algo(ids))
btn_ids.grid(column=0, row=4, sticky="ew", padx=10, pady=5)

btn_ucs = tk.Button(frm_algo_options, text="UCS", bg="white", font=btn_font, command=lambda: run_and_animate_algo(ucs))
btn_ucs.grid(column=0, row=5, sticky="ew", padx=10, pady=5)

btn_ucs = tk.Button(frm_algo_options, text="GREEDY", bg="white", font=btn_font, command=lambda: run_and_animate_algo(greedy))
btn_ucs.grid(column=0, row=6, sticky="ew", padx=10, pady=5)

btn_ucs = tk.Button(frm_algo_options, text="A_STAR", bg="white", font=btn_font, command=lambda: run_and_animate_algo(a_star))
btn_ucs.grid(column=0, row=7, sticky="ew", padx=10, pady=5)

# Algorithm buttons and Randomize button spacing
frm_algo_options.rowconfigure(8, weight=1) 

btn_random = tk.Button(frm_algo_options, text="Randomize", bg="white", font=btn_font, command=randomize_map)
btn_random.grid(column=0, row=9, sticky="ew", padx=10, pady=10)

btn_clr_log = tk.Button(frm_algo_options, text="Clear log", bg="white", font=btn_font, command=lambda: update_log(clear=True))
btn_clr_log.grid(column=0, row=10, sticky="ew", padx=10, pady=10)

# --- LOG WIDGET SETUP ---  
log_text = tk.Text(frm_state_log, bg="white", wrap=tk.WORD, font=("Consolas", 10))
log_scroll = tk.Scrollbar(frm_state_log, command=log_text.yview)
log_text.configure(yscrollcommand=log_scroll.set)

log_scroll.pack(side=tk.RIGHT, fill=tk.Y)
log_text.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

window.mainloop()